In [ ]:

pip install pandas numpy matplotlib scikit-learn ta openpyxl

In [ ]:




















































































































































































































































"""
=============================================================================
  MAROC TELECOM (IAM) — Analyse Technique & Prédictions par Régression
  Bourse de Casablanca (BVC) | Devise: MAD
=============================================================================
Indicateurs : SMA, EMA, Bollinger, RSI, MACD, Stochastique, ATR, OBV,
              Williams %R, CCI
Modèles     : Linéaire, Polynomiale deg2/deg3, Ridge, Lasso, SVR,
              Random Forest, Gradient Boosting
=============================================================================
"""
import warnings
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use('Agg')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import dates as mdates

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline

# ─────────────────────────────────────────────────────────────
# 1. CHARGEMENT & NETTOYAGE
# ─────────────────────────────────────────────────────────────
FILE_PATH = "Maroc_Telecom_IAM_Cours_Journaliers.xlsx"

df = pd.read_excel(FILE_PATH, sheet_name='Cours Journaliers', header=2)
df.columns = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Change_pct', 'MarketCap']
df['Date'] = pd.to_datetime(df['Date'])
df = df.dropna(subset=['Close', 'Open', 'High', 'Low']).sort_values('Date').reset_index(drop=True)
df['Volume'] = pd.to_numeric(df['Volume'], errors='coerce').fillna(0)

print("=" * 65)
print("  MAROC TELECOM (IAM) — Analyse Boursière Complète")
print("=" * 65)
print(f"  Période  : {df['Date'].min().date()}  →  {df['Date'].max().date()}")
print(f"  Nb jours : {len(df)} séances")
print(f"  Clôture  : min={df['Close'].min():.2f}  max={df['Close'].max():.2f}  moy={df['Close'].mean():.2f} MAD")
print("=" * 65)

# ─────────────────────────────────────────────────────────────
# 2. INDICATEURS TECHNIQUES
# ─────────────────────────────────────────────────────────────
def sma(s, n):
    return s.rolling(n).mean()

def ema(s, n):
    return s.ewm(span=n, adjust=False).mean()

def rsi(s, n=14):
    d = s.diff()
    g = d.clip(lower=0).rolling(n).mean()
    l = (-d.clip(upper=0)).rolling(n).mean()
    return 100 - 100 / (1 + g / l.replace(0, np.nan))

def macd_calc(s, fast=12, slow=26, sig=9):
    m = ema(s, fast) - ema(s, slow)
    sg = ema(m, sig)
    return m, sg, m - sg

def bollinger(s, n=20, k=2):
    mid = sma(s, n)
    std = s.rolling(n).std()
    return mid + k * std, mid, mid - k * std

def stochastic(h, l, c, k=14, d=3):
    lo = l.rolling(k).min()
    hi = h.rolling(k).max()
    pk = 100 * (c - lo) / (hi - lo).replace(0, np.nan)
    return pk, pk.rolling(d).mean()

def atr_calc(h, l, c, n=14):
    tr = pd.concat([(h - l), (h - c.shift()).abs(), (l - c.shift()).abs()], axis=1).max(axis=1)
    return tr.rolling(n).mean()

def obv_calc(c, v):
    return (np.sign(c.diff()).fillna(0) * v).cumsum()

def williams_r(h, l, c, n=14):
    hi = h.rolling(n).max()
    lo = l.rolling(n).min()
    return -100 * (hi - c) / (hi - lo).replace(0, np.nan)

def cci_calc(h, l, c, n=20):
    tp = (h + l + c) / 3
    ma = tp.rolling(n).mean()
    md = tp.rolling(n).apply(lambda x: np.mean(np.abs(x - x.mean())), raw=True)
    return (tp - ma) / (0.015 * md.replace(0, np.nan))

df['SMA20']    = sma(df['Close'], 20)
df['SMA50']    = sma(df['Close'], 50)
df['SMA200']   = sma(df['Close'], 200)
df['EMA12']    = ema(df['Close'], 12)
df['EMA26']    = ema(df['Close'], 26)
df['RSI']      = rsi(df['Close'])
df['MACD'], df['Signal'], df['Hist'] = macd_calc(df['Close'])
df['BB_up'], df['BB_mid'], df['BB_dn'] = bollinger(df['Close'])
df['Stoch_K'], df['Stoch_D'] = stochastic(df['High'], df['Low'], df['Close'])
df['ATR']      = atr_calc(df['High'], df['Low'], df['Close'])
df['OBV']      = obv_calc(df['Close'], df['Volume'])
df['WillR']    = williams_r(df['High'], df['Low'], df['Close'])
df['CCI']      = cci_calc(df['High'], df['Low'], df['Close'])
df['Return']   = df['Close'].pct_change()

print("✔ Indicateurs techniques calculés avec succès.\n")

# ─────────────────────────────────────────────────────────────
# 3. PRÉPARATION ML  (dropna uniquement sur les features ML)
# ─────────────────────────────────────────────────────────────
FEATURES = ['SMA20', 'SMA50', 'EMA12', 'RSI', 'MACD', 'BB_up', 'BB_dn', 'ATR', 'Stoch_K']
df_ml = df.dropna(subset=FEATURES + ['Return']).copy().reset_index(drop=True)
df_ml['t'] = np.arange(len(df_ml))

feat_cols = ['t'] + FEATURES
X = df_ml[feat_cols].values
y = df_ml['Close'].values
dates_ml = df_ml['Date'].values

scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X)

# ─────────────────────────────────────────────────────────────
# 4. MODÈLES DE RÉGRESSION
# ─────────────────────────────────────────────────────────────
HORIZON = 60

models = {
    'Régression Linéaire': LinearRegression(),
    'Polynomiale deg 2':   Pipeline([('poly', PolynomialFeatures(2, include_bias=False)),
                                     ('lr',   LinearRegression())]),
    'Polynomiale deg 3':   Pipeline([('poly', PolynomialFeatures(3, include_bias=False)),
                                     ('lr',   LinearRegression())]),
    'Ridge (α=1)':         Ridge(alpha=1.0),
    'Lasso (α=0.1)':       Lasso(alpha=0.1, max_iter=10000),
    'SVR (RBF)':           SVR(kernel='rbf', C=100, epsilon=0.1),
    'Random Forest':       RandomForestRegressor(n_estimators=200, random_state=42),
    'Gradient Boosting':   GradientBoostingRegressor(n_estimators=200, random_state=42),
}

results = {}
print(f"{'Modèle':<25} {'R²':>8} {'MAE':>10} {'RMSE':>10} {'Tendance 60j':>15}")
print("-" * 72)

for name, model in models.items():
    model.fit(X_scaled, y)
    y_pred = model.predict(X_scaled)
    r2   = r2_score(y, y_pred)
    mae  = mean_absolute_error(y, y_pred)
    rmse = np.sqrt(mean_squared_error(y, y_pred))

    t_last = df_ml['t'].max()
    future_t = np.arange(t_last + 1, t_last + 1 + HORIZON)
    Xf_raw = np.column_stack(
        [future_t] + [np.full(HORIZON, X[-1, i]) for i in range(1, len(feat_cols))]
    )
    Xf = scaler_X.transform(Xf_raw)
    y_future = model.predict(Xf)

    trend_pct = (y_future[-1] - y_pred[-1]) / y_pred[-1] * 100
    direction = "↑ Haussière" if trend_pct > 0 else "↓ Baissière"
    print(f"{name:<25} {r2:>8.4f} {mae:>10.4f} {rmse:>10.4f} {trend_pct:>+8.2f}% {direction}")

    results[name] = dict(model=model, y_pred=y_pred, y_future=y_future,
                         r2=r2, mae=mae, rmse=rmse, trend_pct=trend_pct)

best_name = max(results, key=lambda k: results[k]['r2'])
best = results[best_name]
print(f"\n★  Meilleur modèle : {best_name}  (R² = {best['r2']:.4f})\n")

last_date    = pd.Timestamp(dates_ml[-1])
future_dates = pd.bdate_range(start=last_date + pd.offsets.BDay(1), periods=HORIZON)

# ─────────────────────────────────────────────────────────────
# 5. SIGNAUX TEXTUELS
# ─────────────────────────────────────────────────────────────
def generate_signals(df):
    last = df.iloc[-1]
    sigs = []
    r = last['RSI']
    sigs.append(f"RSI={r:.1f} → {'SURACHAT : risque de correction.' if r > 70 else 'SURVENTE : signal achat potentiel.' if r < 30 else 'Zone NEUTRE : pas de signal fort.'}")
    sigs.append("MACD > Signal → Dynamique HAUSSIÈRE." if last['MACD'] > last['Signal'] else "MACD < Signal → Pression VENDEUSE présente.")
    c = last['Close']
    sigs.append("Prix > Bande Bollinger sup. → SURACHAT." if c > last['BB_up'] else
                "Prix < Bande Bollinger inf. → SURVENTE." if c < last['BB_dn'] else
                "Prix dans les bandes Bollinger → Volatilité modérée.")
    sigs.append("Close > SMA50 → Tendance HAUSSIÈRE moyen terme." if c > last['SMA50'] else
                "Close < SMA50 → Tendance BAISSIÈRE moyen terme.")
    k = last['Stoch_K']
    sigs.append(f"Stoch K={k:.1f} → {'SURACHAT.' if k > 80 else 'SURVENTE.' if k < 20 else 'Zone neutre.'}")
    cc = last['CCI']
    sigs.append(f"CCI={cc:.1f} → {'Tendance forte HAUSSIÈRE.' if cc > 100 else 'Tendance forte BAISSIÈRE.' if cc < -100 else 'CCI neutre.'}")
    w = last['WillR']
    sigs.append(f"Williams %R={w:.1f} → {'SURACHAT.' if w > -20 else 'SURVENTE.' if w < -80 else 'Zone neutre.'}")
    return sigs

signal_comments = generate_signals(df_ml)
print("─" * 65)
print("  SIGNAUX TECHNIQUES (dernière séance)")
print("─" * 65)
for i, c in enumerate(signal_comments, 1):
    print(f"  {i}. {c}")
print("─" * 65)

# ─────────────────────────────────────────────────────────────
# STYLE GLOBAL
# ─────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0d1117', 'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d',   'axes.labelcolor': '#c9d1d9',
    'xtick.color': '#8b949e',      'ytick.color': '#8b949e',
    'text.color': '#c9d1d9',       'grid.color': '#21262d',
    'grid.linestyle': '--',        'grid.alpha': 0.5,
    'font.family': 'DejaVu Sans',  'font.size': 9,
})
C_CLOSE   = '#58a6ff'
C_SMA20   = '#f0883e'
C_SMA50   = '#56d364'
C_SMA200  = '#f78166'
C_BB      = '#bc8cff'
C_BUY     = '#56d364'
C_SELL    = '#f85149'
C_NEUTRAL = '#f0883e'
C_PRED    = '#ffa657'
C_FUTURE  = '#d2a8ff'

xd = df_ml['Date']

# ─────────────────────────────────────────────────────────────
# FIGURE 1 : Cours + RSI + MACD + Stochastique + Volume
# ─────────────────────────────────────────────────────────────
print("\n⏳ Figure 1 : Analyse technique principale …")
fig1, axes = plt.subplots(5, 1, figsize=(16, 20),
                           gridspec_kw={'height_ratios': [3, 1, 1, 1, 1]})
fig1.suptitle('MAROC TELECOM (IAM) — Analyse Technique Complète',
              fontsize=16, fontweight='bold', color='white', y=0.98)
ax1, ax2, ax3, ax4, ax5 = axes

# Cours + Bollinger + Moyennes
ax1.fill_between(xd, df_ml['BB_dn'], df_ml['BB_up'], alpha=0.12, color=C_BB, label='Bollinger')
ax1.plot(xd, df_ml['BB_up'], lw=0.8, ls='--', color=C_BB, alpha=0.55)
ax1.plot(xd, df_ml['BB_dn'], lw=0.8, ls='--', color=C_BB, alpha=0.55)
ax1.plot(xd, df_ml['Close'],  lw=1.5, color=C_CLOSE,  label='Clôture', zorder=5)
ax1.plot(xd, df_ml['SMA20'],  lw=1.0, color=C_SMA20,  label='SMA 20')
ax1.plot(xd, df_ml['SMA50'],  lw=1.0, color=C_SMA50,  label='SMA 50')
# SMA200 peut avoir des NaN dans df_ml, on trace quand même
sma200_ml = df['SMA200'].reindex(df_ml.index)
if sma200_ml.notna().any():
    ax1.plot(xd, sma200_ml.values, lw=1.0, color=C_SMA200, label='SMA 200')
ax1.set_ylabel('Cours (MAD)')
ax1.legend(loc='upper left', fontsize=8)
ax1.set_title('Cours avec Moyennes Mobiles & Bandes de Bollinger', fontsize=10)
ax1.grid(True)

# Volume
vol_colors = [C_BUY if r >= 0 else C_SELL for r in df_ml['Return']]
ax2.bar(xd, df_ml['Volume'], color=vol_colors, alpha=0.7, width=1)
ax2.set_ylabel('Volume')
ax2.set_title('Volume des échanges (vert=hausse, rouge=baisse)', fontsize=10)
ax2.grid(True)

# RSI
ax3.plot(xd, df_ml['RSI'], color=C_NEUTRAL, lw=1.2, label='RSI(14)')
ax3.axhline(70, color=C_SELL, lw=1, ls='--', alpha=0.7, label='Surachat 70')
ax3.axhline(30, color=C_BUY,  lw=1, ls='--', alpha=0.7, label='Survente 30')
ax3.fill_between(xd, 70, df_ml['RSI'].clip(lower=70),  alpha=0.2, color=C_SELL)
ax3.fill_between(xd, df_ml['RSI'].clip(upper=30), 30,  alpha=0.2, color=C_BUY)
ax3.set_ylim(0, 100)
ax3.set_ylabel('RSI')
ax3.set_title('RSI (14) — Relative Strength Index', fontsize=10)
ax3.legend(loc='upper left', fontsize=8)
ax3.grid(True)

# MACD
ax4.plot(xd, df_ml['MACD'],   color=C_CLOSE, lw=1.2, label='MACD')
ax4.plot(xd, df_ml['Signal'], color=C_SELL,  lw=1.0, ls='--', label='Signal')
hist_colors = [C_BUY if h >= 0 else C_SELL for h in df_ml['Hist']]
ax4.bar(xd, df_ml['Hist'], color=hist_colors, alpha=0.6, width=1)
ax4.axhline(0, color='#8b949e', lw=0.8)
ax4.set_ylabel('MACD')
ax4.set_title('MACD (12,26,9)', fontsize=10)
ax4.legend(loc='upper left', fontsize=8)
ax4.grid(True)

# Stochastique
ax5.plot(xd, df_ml['Stoch_K'], color=C_SMA20, lw=1.2, label='%K (14)')
ax5.plot(xd, df_ml['Stoch_D'], color=C_BB,    lw=1.0, ls='--', label='%D (3)')
ax5.axhline(80, color=C_SELL, lw=1, ls='--', alpha=0.7)
ax5.axhline(20, color=C_BUY,  lw=1, ls='--', alpha=0.7)
ax5.fill_between(xd, 80, df_ml['Stoch_K'].clip(lower=80), alpha=0.2, color=C_SELL)
ax5.fill_between(xd, df_ml['Stoch_K'].clip(upper=20), 20, alpha=0.2, color=C_BUY)
ax5.set_ylim(0, 100)
ax5.set_ylabel('Stoch %K/%D')
ax5.set_title('Oscillateur Stochastique (14,3)', fontsize=10)
ax5.legend(loc='upper left', fontsize=8)
ax5.grid(True)

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

fig1.tight_layout(rect=[0, 0, 1, 0.97])
fig1.savefig('IAM_01_Analyse_Technique.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.close(fig1)
print("  ✔ IAM_01_Analyse_Technique.png")

# ─────────────────────────────────────────────────────────────
# FIGURE 2 : Williams %R, CCI, ATR, OBV
# ─────────────────────────────────────────────────────────────
print("⏳ Figure 2 : Indicateurs complémentaires …")
fig2, (a1, a2, a3, a4) = plt.subplots(4, 1, figsize=(16, 14))
fig2.suptitle('MAROC TELECOM (IAM) — Indicateurs Complémentaires',
              fontsize=15, fontweight='bold', color='white')

a1.plot(xd, df_ml['WillR'], color='#ffa657', lw=1.2)
a1.axhline(-20, color=C_SELL, lw=1, ls='--', label='Surachat −20')
a1.axhline(-80, color=C_BUY,  lw=1, ls='--', label='Survente −80')
a1.fill_between(xd, -20, df_ml['WillR'].clip(upper=-20), alpha=0.2, color=C_SELL)
a1.fill_between(xd, df_ml['WillR'].clip(lower=-80), -80,  alpha=0.2, color=C_BUY)
a1.set_ylim(-100, 0)
a1.set_ylabel('Williams %R')
a1.set_title('Williams %R (14)', fontsize=10)
a1.legend(fontsize=8)
a1.grid(True)

a2.plot(xd, df_ml['CCI'], color='#79c0ff', lw=1.2)
a2.axhline( 100, color=C_SELL, lw=1, ls='--', alpha=0.7)
a2.axhline(-100, color=C_BUY,  lw=1, ls='--', alpha=0.7)
a2.axhline(0, color='#8b949e', lw=0.8, ls=':')
a2.fill_between(xd, 100, df_ml['CCI'].clip(lower=100),   alpha=0.15, color=C_SELL)
a2.fill_between(xd, df_ml['CCI'].clip(upper=-100), -100,  alpha=0.15, color=C_BUY)
a2.set_ylabel('CCI')
a2.set_title('CCI (20) — Commodity Channel Index', fontsize=10)
a2.grid(True)

a3.plot(xd, df_ml['ATR'], color='#ff7b72', lw=1.2)
a3.fill_between(xd, 0, df_ml['ATR'], alpha=0.2, color='#ff7b72')
a3.set_ylabel('ATR (MAD)')
a3.set_title('ATR (14) — Average True Range (Volatilité)', fontsize=10)
a3.grid(True)

a4.plot(xd, df_ml['OBV'] / 1e6, color='#56d364', lw=1.2)
a4.fill_between(xd, 0, df_ml['OBV'] / 1e6, alpha=0.15, color='#56d364')
a4.set_ylabel('OBV (M)')
a4.set_title('OBV — On-Balance Volume', fontsize=10)
a4.grid(True)

for ax in [a1, a2, a3, a4]:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

fig2.tight_layout(rect=[0, 0, 1, 0.96])
fig2.savefig('IAM_02_Indicateurs_Complementaires.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.close(fig2)
print("  ✔ IAM_02_Indicateurs_Complementaires.png")

# ─────────────────────────────────────────────────────────────
# FIGURE 3 : Comparaison des 8 modèles
# ─────────────────────────────────────────────────────────────
print("⏳ Figure 3 : Comparaison modèles …")
fig3, axes3 = plt.subplots(4, 2, figsize=(18, 22))
fig3.suptitle('MAROC TELECOM (IAM) — Comparaison des Modèles de Régression',
              fontsize=15, fontweight='bold', color='white')

for idx, (name, res) in enumerate(results.items()):
    ax = axes3.flatten()[idx]
    ax.plot(xd, df_ml['Close'],         color=C_CLOSE,  lw=1.2, label='Cours réel', zorder=5)
    ax.plot(xd, res['y_pred'],           color=C_PRED,   lw=1.2, ls='--', label='Modèle ajusté')
    ax.plot(future_dates, res['y_future'], color=C_FUTURE, lw=2.0, label=f'Prévision +{HORIZON}j', zorder=6)
    ax.axvline(last_date, color='#8b949e', lw=1, ls=':', alpha=0.7)
    ax.fill_betweenx(
        [df_ml['Close'].min() * 0.97, df_ml['Close'].max() * 1.03],
        last_date, future_dates[-1], alpha=0.06, color=C_FUTURE
    )
    dir_col = C_BUY if res['trend_pct'] > 0 else C_SELL
    ax.set_title(
        f"{name}  |  R²={res['r2']:.4f}  |  RMSE={res['rmse']:.2f}  |  "
        f"{'↑' if res['trend_pct'] > 0 else '↓'} {abs(res['trend_pct']):.1f}%",
        fontsize=9, color=dir_col
    )
    ax.set_ylabel('Cours (MAD)', fontsize=8)
    ax.legend(fontsize=7, loc='upper left')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
    ax.grid(True)

fig3.tight_layout(rect=[0, 0, 1, 0.97])
fig3.savefig('IAM_03_Comparaison_Modeles.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.close(fig3)
print("  ✔ IAM_03_Comparaison_Modeles.png")

# ─────────────────────────────────────────────────────────────
# FIGURE 4 : Prédictions futures — meilleur modèle
# ─────────────────────────────────────────────────────────────
print("⏳ Figure 4 : Prédictions futures …")
fig4, ax = plt.subplots(figsize=(18, 9))
fig4.patch.set_facecolor('#0d1117')

pf = best['y_future']
ax.fill_between(future_dates, pf * 0.90, pf * 1.10, alpha=0.10, color=C_FUTURE, label='Intervalle ±10%')
ax.fill_between(future_dates, pf * 0.95, pf * 1.05, alpha=0.20, color=C_FUTURE, label='Intervalle ±5%')
ax.plot(xd[-120:], df_ml['Close'].iloc[-120:],   color=C_CLOSE,  lw=2.0, label='Historique (120j)', zorder=6)
ax.plot(xd[-120:], best['y_pred'][-120:],        color=C_PRED,   lw=1.2, ls='--', alpha=0.8, label='Modèle ajusté')
ax.plot(future_dates, pf,                        color=C_FUTURE, lw=2.5, label=f'Prévision {HORIZON}j', zorder=7)
ax.axvline(last_date, color='#8b949e', lw=1.5, ls='--', label=f"Aujourd'hui ({last_date.date()})")

last_close = df_ml['Close'].iloc[-1]
ax.annotate(
    f"Actuel\n{last_close:.2f} MAD",
    xy=(last_date, last_close),
    xytext=(last_date - pd.Timedelta(days=18), last_close * 1.025),
    fontsize=9, color='white', fontweight='bold',
    arrowprops=dict(arrowstyle='->', color='white', lw=1.2)
)

target    = pf[-1]
t_col     = C_BUY if target > last_close else C_SELL
ax.annotate(
    f"Cible J+{HORIZON}\n{target:.2f} MAD\n({(target - last_close) / last_close * 100:+.2f}%)",
    xy=(future_dates[-1], target),
    xytext=(future_dates[-1] - pd.Timedelta(days=12), target * 1.03),
    fontsize=9, color=t_col, fontweight='bold',
    arrowprops=dict(arrowstyle='->', color=t_col, lw=1.2)
)

ax.set_title(
    f"Prédictions futures — {best_name} (R²={best['r2']:.4f})  |  Horizon : {HORIZON} jours ouvrés",
    fontsize=12, fontweight='bold', pad=12
)
ax.set_ylabel('Cours (MAD)', fontsize=10)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b %Y'))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=35, ha='right')
ax.legend(fontsize=9)
ax.grid(True)
fig4.tight_layout()
fig4.savefig('IAM_04_Predictions_Futures.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.close(fig4)
print("  ✔ IAM_04_Predictions_Futures.png")

# ─────────────────────────────────────────────────────────────
# FIGURE 5 : Tableau de bord
# ─────────────────────────────────────────────────────────────
print("⏳ Figure 5 : Tableau de bord …")
fig5, axes5 = plt.subplots(2, 3, figsize=(18, 12))
fig5.suptitle('MAROC TELECOM (IAM) — Tableau de Bord des Signaux',
              fontsize=15, fontweight='bold', color='white')

short_names = [n.replace('Régression ', '').replace('Polynomiale ', 'Poly ') for n in results]
r2_vals     = [results[n]['r2']       for n in results]
rmse_vals   = [results[n]['rmse']     for n in results]
trend_vals  = [results[n]['trend_pct'] for n in results]

# R²
ax = axes5[0, 0]
c_r2 = [C_BUY if v > 0.9 else C_NEUTRAL if v > 0.7 else C_SELL for v in r2_vals]
bars = ax.barh(short_names, r2_vals, color=c_r2, edgecolor='none', height=0.6)
ax.axvline(0.9, color='white', lw=1, ls='--', alpha=0.5)
for bar, val in zip(bars, r2_vals):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=8, color='white')
ax.set_xlim(0, 1.05)
ax.set_xlabel('Score R²')
ax.set_title('Performance R² par Modèle', fontsize=10)
ax.grid(axis='x')

# RMSE
ax = axes5[0, 1]
ax.bar(short_names, rmse_vals, color='#58a6ff', alpha=0.8, edgecolor='none')
ax.set_ylabel('RMSE (MAD)')
ax.set_title('RMSE par Modèle', fontsize=10)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right', fontsize=7)
ax.grid(axis='y')

# Tendances
ax = axes5[0, 2]
c_tr = [C_BUY if t > 0 else C_SELL for t in trend_vals]
ax.bar(short_names, trend_vals, color=c_tr, alpha=0.85, edgecolor='none')
ax.axhline(0, color='white', lw=1)
ax.set_ylabel(f'Variation prévue ({HORIZON}j) (%)')
ax.set_title(f'Tendance à {HORIZON}j', fontsize=10)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right', fontsize=7)
ax.grid(axis='y')

# Distribution rendements
ax = axes5[1, 0]
rets = df_ml['Return'].dropna() * 100
ax.hist(rets, bins=50, color=C_CLOSE, edgecolor='none', alpha=0.8)
ax.axvline(rets.mean(), color=C_BUY, lw=2, ls='--', label=f'Moy={rets.mean():.3f}%')
ax.axvline(0, color='white', lw=1, alpha=0.5)
ax.set_xlabel('Rendement (%)')
ax.set_ylabel('Fréquence')
ax.set_title('Distribution des Rendements Journaliers', fontsize=10)
ax.legend(fontsize=8)
ax.grid(True)

# Rendement mensuel
ax = axes5[1, 1]
df_ml_tmp = df_ml.copy()
df_ml_tmp['YM'] = df_ml_tmp['Date'].dt.to_period('M')
monthly = df_ml_tmp.groupby('YM')['Return'].mean() * 100
months_str = [str(p) for p in monthly.index]
c_m = [C_BUY if v > 0 else C_SELL for v in monthly.values]
ax.bar(range(len(months_str)), monthly.values, color=c_m, edgecolor='none', alpha=0.85)
ax.axhline(0, color='white', lw=1)
ax.set_xticks(range(len(months_str)))
ax.set_xticklabels(months_str, rotation=75, ha='right', fontsize=6)
ax.set_ylabel('Rdt moy (%)')
ax.set_title('Rendement Mensuel Moyen', fontsize=10)
ax.grid(axis='y')

# Tableau valeurs clés
ax = axes5[1, 2]
ax.axis('off')
last = df_ml.iloc[-1]
sma200_val = df_ml['SMA200'].dropna().iloc[-1] if df_ml['SMA200'].notna().any() else None
signal_data = [
    ('Clôture',     f"{last['Close']:.2f} MAD",  'white'),
    ('RSI (14)',    f"{last['RSI']:.1f}",         C_SELL if last['RSI'] > 70 else C_BUY if last['RSI'] < 30 else C_NEUTRAL),
    ('MACD',        f"{last['MACD']:.3f}",        C_BUY if last['MACD'] > last['Signal'] else C_SELL),
    ('Signal MACD', f"{last['Signal']:.3f}",      'white'),
    ('Boll. Sup.',  f"{last['BB_up']:.2f}",       C_SELL),
    ('Boll. Inf.',  f"{last['BB_dn']:.2f}",       C_BUY),
    ('SMA 20',      f"{last['SMA20']:.2f}",       C_SMA20),
    ('SMA 50',      f"{last['SMA50']:.2f}",       C_SMA50),
    ('SMA 200',     f"{sma200_val:.2f}" if sma200_val else 'N/A', C_SMA200),
    ('Stoch K',     f"{last['Stoch_K']:.1f}",     C_SELL if last['Stoch_K'] > 80 else C_BUY if last['Stoch_K'] < 20 else C_NEUTRAL),
    ('ATR (14)',    f"{last['ATR']:.2f}",          '#ffa657'),
    ('Williams %R', f"{last['WillR']:.1f}",       C_SELL if last['WillR'] > -20 else C_BUY if last['WillR'] < -80 else C_NEUTRAL),
    ('CCI (20)',    f"{last['CCI']:.1f}",          C_SELL if last['CCI'] > 100 else C_BUY if last['CCI'] < -100 else C_NEUTRAL),
]
ax.set_title(f"Valeurs clés — {last['Date'].date()}", fontsize=10, pad=8)
for i, (label, val, col) in enumerate(signal_data):
    y_pos = 1.0 - i * 0.075
    ax.text(0.05, y_pos, label, transform=ax.transAxes, fontsize=9, color='#8b949e', va='top')
    ax.text(0.65, y_pos, val,   transform=ax.transAxes, fontsize=9, color=col, va='top', fontweight='bold')

fig5.tight_layout(rect=[0, 0, 1, 0.96])
fig5.savefig('IAM_05_Tableau_de_Bord.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.close(fig5)
print("  ✔ IAM_05_Tableau_de_Bord.png")

# ─────────────────────────────────────────────────────────────
# FIGURE 6 : Rapport de synthèse
# ─────────────────────────────────────────────────────────────
print("⏳ Figure 6 : Rapport de synthèse …")
fig6 = plt.figure(figsize=(16, 12))
fig6.patch.set_facecolor('#0d1117')
ax6 = fig6.add_subplot(111)
ax6.axis('off')

last       = df_ml.iloc[-1]
target_val = best['y_future'][-1]
trend_best = best['trend_pct']
dir_str    = "HAUSSIÈRE" if trend_best > 0 else "BAISSIÈRE"
dir_col    = C_BUY if trend_best > 0 else C_SELL

title_text = (
    f"MAROC TELECOM (IAM) — Synthèse & Rapport d'Analyse\n"
    f"Date : {last['Date'].date()}  |  Clôture : {last['Close']:.2f} MAD\n"
    f"Modèle : {best_name}  |  R² = {best['r2']:.4f}"
)

rapport = [
    ("RÉSUMÉ EXÉCUTIF", C_BB, 13, True),
    ("", 'white', 9, False),
    (f"Le titre IAM (Maroc Telecom) clôture à {last['Close']:.2f} MAD.", 'white', 10, False),
    (f"Meilleur modèle : {best_name}  —  R²={best['r2']:.4f}  |  RMSE={best['rmse']:.2f} MAD  |  MAE={best['mae']:.2f} MAD.", 'white', 10, False),
    (f"Prévision à {HORIZON} jours : tendance {dir_str} → Cible estimée {target_val:.2f} MAD ({trend_best:+.2f}%).", dir_col, 10, True),
    ("", 'white', 9, False),
    ("SIGNAUX TECHNIQUES", C_BB, 12, True),
    ("", 'white', 8, False),
] + [
    (f"  • {c}", 'white', 9, False) for c in signal_comments
] + [
    ("", 'white', 9, False),
    ("STATISTIQUES DE RISQUE", C_BB, 12, True),
    ("", 'white', 8, False),
    (f"  • Volatilité ATR(14) : {last['ATR']:.2f} MAD  — "
     + ("Élevée : gestion du risque recommandée." if last['ATR'] > 2 else "Faible : marché calme."),
     '#ffa657', 9, False),
    (f"  • Rendement moy. journalier : {df_ml['Return'].mean()*100:.4f}%  "
     f"| Volatilité annualisée : {df_ml['Return'].std()*np.sqrt(252)*100:.2f}%",
     'white', 9, False),
    (f"  • Gain max 1j : {df_ml['Return'].max()*100:.2f}%  "
     f"|  Perte max 1j : {df_ml['Return'].min()*100:.2f}%",
     'white', 9, False),
    ("", 'white', 9, False),
    ("AVERTISSEMENT", '#8b949e', 10, True),
    ("  Analyse automatique à des fins éducatives uniquement.", '#8b949e', 8, False),
    ("  Ne constitue pas un conseil en investissement.", '#8b949e', 8, False),
    ("  Tout investissement comporte des risques de perte en capital.", '#8b949e', 8, False),
]

ax6.text(0.5, 0.97, title_text, transform=ax6.transAxes,
         fontsize=12, fontweight='bold', color='white', ha='center', va='top',
         bbox=dict(boxstyle='round,pad=0.6', facecolor='#161b22', edgecolor='#30363d', lw=1.5))

y_cur = 0.82
for (text, col, fs, bold) in rapport:
    ax6.text(0.03, y_cur, text, transform=ax6.transAxes, fontsize=fs,
             color=col, va='top', fontweight='bold' if bold else 'normal')
    y_cur -= 0.033

fig6.tight_layout()
fig6.savefig('IAM_06_Rapport_Synthese.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.close(fig6)
print("  ✔ IAM_06_Rapport_Synthese.png")

# ─────────────────────────────────────────────────────────────
# RÉSUMÉ FINAL CONSOLE
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("  RÉSUMÉ FINAL — MAROC TELECOM (IAM)")
print("=" * 65)
print(f"  Cours actuel    : {df_ml['Close'].iloc[-1]:.2f} MAD")
print(f"  Meilleur modèle : {best_name}")
print(f"  R²              : {best['r2']:.4f}")
print(f"  RMSE            : {best['rmse']:.4f} MAD")
print(f"  MAE             : {best['mae']:.4f} MAD")
print(f"  Prévision J+{HORIZON}  : {best['y_future'][-1]:.2f} MAD ({best['trend_pct']:+.2f}%)")
print("─" * 65)
labels = ['Analyse_Technique', 'Indicateurs_Complementaires', 'Comparaison_Modeles',
          'Predictions_Futures', 'Tableau_de_Bord', 'Rapport_Synthese']
print("  FICHIERS GÉNÉRÉS :")
for i, lbl in enumerate(labels, 1):
    print(f"    IAM_0{i}_{lbl}.png")
print("=" * 65)



  MAROC TELECOM (IAM) — Analyse Boursière Complète
  Période  : 2023-01-02  →  2025-12-31
  Nb jours : 781 séances
  Clôture  : min=93.06  max=139.48  moy=113.86 MAD
✔ Indicateurs techniques calculés avec succès.

Modèle                          R²        MAE       RMSE    Tendance 60j
------------------------------------------------------------------------
Régression Linéaire         0.9959     0.5116     0.7634    -0.01% ↓ Baissière
Polynomiale deg 2           0.9986     0.3514     0.4526    -0.01% ↓ Baissière
Polynomiale deg 3           0.9995     0.2130     0.2731    +0.52% ↑ Haussière
Ridge (α=1)                 0.9958     0.5117     0.7671    -0.00% ↓ Baissière
Lasso (α=0.1)               0.9954     0.5191     0.8057    +0.00% ↓ Baissière
SVR (RBF)                   0.9996     0.1696     0.2452    +0.20% ↑ Haussière
Random Forest               0.9994     0.1701     0.2792    +0.00% ↓ Baissière
Gradient Boosting           0.9996     0.1884     0.2449    +0.00% ↓ Baissière

★  Meil